In [1]:
import re
import pandas as pd

In [2]:
csv_file = "breast_cancer/confidence_rank_source.csv"

new_edges_csv = "breast_cancer/confidence_rank_new_edges.csv"
augmented_csv = "breast_cancer/confidence_rank_augmented_temp.csv"

In [3]:
# Load your CSV file
df = pd.read_csv(csv_file)

In [4]:
# Ensure all relevant columns are strings
for col in ["Regulator", "Target", "Sign", "Model", "Classification", "Constraint", "Notes"]:
    df[col] = df[col].astype(str)

In [5]:
def get_base_and_level(node):
    """
    A     -> ("A", 1)
    A_2   -> ("A", 2)
    A_17  -> ("A", 17)
    """
    match = re.match(r"^(.*)_(\d+)$", str(node))

    if match:
        return match.group(1), int(match.group(2))

    return str(node), 1

In [6]:
for x in ["A", "A_2", "A_3", "B"]:
    print(x, get_base_and_level(x))

A ('A', 1)
A_2 ('A', 2)
A_3 ('A', 3)
B ('B', 1)


In [7]:
def get_possible_levels(df):
    possible = {}

    nodes = pd.concat([
        df["Regulator"],
        df["Target"]
    ]).dropna().unique()

    for node in nodes:
        base, level = get_base_and_level(node)
        possible.setdefault(base, set()).add(level)

    return possible

In [8]:
possible_levels = get_possible_levels(df)

print(possible_levels)

{'AKT': {1}, 'EGFR': {1}, 'HER2': {1}, 'HER3': {1, 2}, 'PTEN': {1, 2}, 'CyclinB1': {1}, 'p53': {1}, 'CyclinD': {1, 2}, 'ERK': {1, 2}, 'GSK3B': {1}, 'RPS6': {1}, 'erlotinib': {1}, 'pertuzumab': {1}, 'stimulus': {1}, 'trastuzumab': {1}, 'RAF1': {1, 2}, 'FOXO1FOXO3': {1}, 'PRAS40': {1}, 'NFKB': {1}, 'RB': {1}, 'S6K': {1}, 'TSC2': {1}, 'JUN': {1}, 'p38': {1}, 'PDK1_pm': {1}, 'mTORC1': {1}, 'EGF': {1}, 'PLCg': {1}, 'NRG1': {1}, 'MEK': {1, 2}, 'PKCa': {1}, 'BAX': {1}, 'Ipatasertib': {1}, 'PIP3': {1, 2}, 'mTORC2_pm': {1}, 'Apoptosis': {1, 2, 3}, 'BAD': {1}, 'BCL2': {1}, 'BIM': {1}, 'MCL1': {1}, 'MAPK': {1, 2}, 'PIM': {1}, 'BCL2_T': {1}, 'ER_transcription': {1, 2}, 'BIM_T': {1}, 'FOXO3': {1}, 'Palbociclib': {1}, 'E2F': {1, 2, 3}, 'pRB': {1, 2, 3}, 'ER': {1}, 'ESR1': {1, 2}, 'FOXA1': {1}, 'KMT2D': {1}, 'PBX1': {1}, 'Fulvestrant': {1}, 'FOXO3_Ub': {1}, 'SGK1': {1}, 'HER2HER3c': {1, 2}, 'Neratinib': {1}, 'HER3_T': {1}, 'IGF1R': {1, 2}, 'IGF1R_T': {1, 2}, 'RAS': {1, 2, 3}, 'Trametinib': {1}, 'Tran

In [9]:
def get_model_levels(model_df):
    model_levels = {}

    nodes = pd.concat([model_df["Regulator"],model_df["Target"]]).dropna().unique()

    for node in nodes:
        base, level = get_base_and_level(node)
        model_levels.setdefault(base, set()).add(level)

    return model_levels

In [10]:
def get_closest_level(levels, requested_level):
    """
    Return the highest available level <= requested_level.
    """
    return max(
        level
        for level in levels
        if level <= requested_level
    )

In [11]:
def set_edge_metadata(new_row, reg_base, reg_level, tar_base, tar_level,
                      source_row, source_reg, source_tar, model):

    if reg_base == tar_base:

        if reg_level < tar_level:
            new_row["Confidence rank"] = "0"
            new_row["Constraint"] = "necessary"
            new_row["Classification"] = ""
            new_row["Notes"] = (
                "Reaching a higher level necessitates "
                "achieving the lower level first."
            )

        elif reg_level > tar_level:
            new_row["Confidence rank"] = "-"
            new_row["Constraint"] = ""
            new_row["Classification"] = "unsupported"
            new_row["Notes"] = (
                "High-level should not activate low-level."
            )

    else:
        new_row["Constraint"] = ""
        orig_note = (
            ""
            if pd.isna(source_row["Notes"])
            else str(source_row["Notes"])
        )

        new_row["Notes"] = (
            f"Duplicated from original edge "
            f"({source_reg} -> {source_tar}, Model={model}). "
            f"Original note: {orig_note}"
        )

    return new_row

In [12]:
def generate_edges_for_model(model_df, possible_levels):

    model = model_df["Model"].iloc[0]

    model_levels = get_model_levels(model_df)

    original_edges = set(zip(model_df["Regulator"], model_df["Target"]))

    new_rows = []

    for _, source_row in model_df.iterrows():

        source_reg = source_row["Regulator"]
        source_tar = source_row["Target"]

        reg_base, _ = get_base_and_level(source_reg)
        tar_base, _ = get_base_and_level(source_tar)

        for reg_level in possible_levels.get(reg_base, set()):
            for tar_level in possible_levels.get(tar_base, set()):

                # ---------------------------------------------
                # New nodes only
                # ---------------------------------------------

                reg_is_new = (reg_level not in model_levels[reg_base])
                tar_is_new = (tar_level not in model_levels[tar_base])

                if not (reg_is_new or tar_is_new):
                    continue

                # ---------------------------------------------
                # Construct requested edge
                # ---------------------------------------------

                new_reg = (
                    reg_base
                    if reg_level == 1
                    else f"{reg_base}_{reg_level}"
                )

                new_tar = (
                    tar_base
                    if tar_level == 1
                    else f"{tar_base}_{tar_level}"
                )

                # ---------------------------------------------
                # Find closest original levels
                # ---------------------------------------------

                source_reg_level = get_closest_level(model_levels[reg_base],reg_level)

                source_tar_level = get_closest_level(model_levels[tar_base],tar_level)

                source_reg = (
                    reg_base
                    if source_reg_level == 1
                    else f"{reg_base}_{source_reg_level}"
                )

                source_tar = (
                    tar_base
                    if source_tar_level == 1
                    else f"{tar_base}_{source_tar_level}"
                )

                # ---------------------------------------------
                # Source edge must exist
                # ---------------------------------------------

                if (source_reg, source_tar) not in original_edges:
                    continue

                # ---------------------------------------------
                # Target edge must not already exist
                # ---------------------------------------------

                if (new_reg, new_tar) in original_edges:
                    continue

                # ---------------------------------------------
                # Copy source edge
                # ---------------------------------------------

                new_row = source_row.copy()
                new_row["Regulator"] = new_reg
                new_row["Target"] = new_tar

                # ---------------------------------------------
                # Set metadata
                # ---------------------------------------------

                new_row = set_edge_metadata(
                    new_row,
                    reg_base,
                    reg_level,
                    tar_base,
                    tar_level,
                    source_row,
                    source_reg,
                    source_tar,
                    model
                )

                new_rows.append(new_row)

    return pd.DataFrame(new_rows)

In [13]:
possible_levels = get_possible_levels(df)

all_new_rows = []

for model, model_df in df.groupby("Model"):

    if model == "merge":
        continue

    model_new_edges = generate_edges_for_model(
        model_df,
        possible_levels
    )

    all_new_rows.append(model_new_edges)


new_edges = pd.concat(all_new_rows,ignore_index=True)

new_edges["index"] = range(len(df) + 1,len(df) + len(new_edges) + 1)

df_augmented = (pd.concat([df, new_edges], ignore_index=True).drop_duplicates())

In [14]:
print(f"Original edges : {len(df)}")
print(f"New edges      : {len(new_edges)}")
print(f"Total edges    : {len(df_augmented)}")

Original edges : 1667
New edges      : 644
Total edges    : 2311


In [15]:
# Save only the newly created edges
new_edges.to_csv(new_edges_csv, index=False)

In [16]:
# Count total edges
n_original = len(df)
n_new = len(new_edges)
n_total = len(df_augmented)

# Count unique regulator-target pairs
n_unique_original = len(df[["Regulator", "Target"]].drop_duplicates())
n_unique_new = len(new_edges[["Regulator", "Target"]].drop_duplicates())
n_unique_total = len(df_augmented[["Regulator", "Target"]].drop_duplicates())

print(f"Original edges          : {n_original}")
print(f"New edges               : {n_new}")
print(f"Total edges             : {n_total}")
print()
print(f"Unique original edges   : {n_unique_original}")
print(f"Unique new edges        : {n_unique_new}")
print(f"Unique total edges      : {n_unique_total}")

Original edges          : 1667
New edges               : 644
Total edges             : 2311

Unique original edges   : 697
Unique new edges        : 236
Unique total edges      : 885


In [17]:
cols_to_check = ["Sign", "Confidence rank", "Constraint", "Classification"]

conflicts = []

for (reg, tar), group in df_augmented.groupby(["Regulator", "Target"]):
    differing = {}

    for col in cols_to_check:
        values = group[col].fillna("").unique()
        if len(values) > 1:
            differing[col] = list(values)

    if differing:
        conflicts.append({
            "Regulator": reg,
            "Target": tar,
            "Conflicting columns": differing
        })

print(f"Found {len(conflicts)} regulator-target pairs with inconsistencies.\n")

for conflict in conflicts:
    print(f"{conflict['Regulator']} -> {conflict['Target']}")
    for col, values in conflict["Conflicting columns"].items():
        print(f"  {col}: {values}")
    print()

Found 6 regulator-target pairs with inconsistencies.

CyclinB1 -> AKT
  Sign: ['positive', 'negative']
  Confidence rank: ['2', '-']
  Classification: ['indirect', 'unsupported']

E2F_3 -> cycE_CDK2_T
  Confidence rank: ['2', '1']
  Classification: ['indirect', 'direct']

GSK3B -> TSC2
  Sign: ['negative', 'positive']
  Confidence rank: ['-', '1']
  Constraint: ['', 'regulates']
  Classification: ['unsupported', 'direct']

RAS_2 -> RAF1
  Constraint: ['regulates', '']

cycE_CDK2 -> pRB_3
  Constraint: ['regulates', '']

p53 -> RB
  Sign: ['positive', 'negative']
  Confidence rank: ['2', '-']
  Classification: ['indirect', 'unsupported']



In [18]:
df_augmented = df_augmented.sort_values(by=list(df_augmented.columns))
df_augmented.to_csv(augmented_csv, index=False)